# FedSwarm — Phase 2.1 centralized ceiling (Colab GPU)

Trains the centralized performance ceiling every FL method in later phases gets measured against: `SimpleCNN` (primary config, 112px) across 5 seeds x {groupnorm, batchnorm}, plus an optional secondary `ResNet-18` @224 table.

**Before running:**
1. **Turn on GPU.** Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4). No account verification needed, unlike Kaggle.
2. **Have `archive (2).zip` (the Brain Tumor MRI dataset, ~164MB) ready to upload** -- the same file used to build the manifest locally. The cell below prompts an upload dialog; point it at that file from your Downloads folder.

This notebook does **not** need `flwr` -- that's only required once the Flower client/server harness exists (Phase 3+). Centralized training is plain PyTorch.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/content/ResearchPaper"

# Colab sessions can survive a cell re-run (e.g. retrying after an earlier cell
# failed) without wiping /content, so a plain `git clone` here fails with exit
# code 128 ("destination path already exists and is not an empty directory") on a
# rerun. Make this idempotent: pull if it's already a clone of this repo, re-clone
# if the directory exists but isn't (a partial/failed prior clone), else clone.
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # A force-push (e.g. a rebased history) makes --ff-only fail even on a
    # legitimate clone of this repo; re-clone rather than hard-failing the cell.
    pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    if pull.returncode != 0:
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
elif os.path.isdir(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

In [ ]:
%cd /content/ResearchPaper

# Colab's base image already has torch/torchvision/numpy/pandas/scikit-learn/
# scipy/matplotlib/pillow/tqdm. Only these are missing for centralized training
# (flwr/flwr-datasets/imagehash/kaggle are not imported by this training path).
!pip install -q omegaconf rich

# Install fedswarm itself (editable, no deps -- everything it needs is already
# satisfied above). Every step below shells out to a fresh interpreter via
# `!python -m fedswarm...` or `subprocess.run(["python", ...])`, and none of those
# inherit this kernel's in-memory sys.path -- so without a real install they all
# fail with `ModuleNotFoundError: No module named 'fedswarm'`. That is what
# silently broke the dataset download, cache build, and training steps on every
# prior Colab run.
!pip install -q -e . --no-deps

## Stage checkpoints (run this first)

Colab's free tier recycles the runtime when the GPU quota runs out, wiping all of
`/content` without warning. Each expensive stage is therefore mirrored to Drive as
soon as it completes, and every cell below skips itself when its stage is already
satisfied:

| stage | cost if lost | restoring it skips |
|---|---|---|
| `dataset` | a 164MB upload | the upload cell |
| `cache` | JPEG decoding, and needs `dataset` | upload **and** extraction |
| `results` | GPU-hours | every finished run |

So a resumed session normally restores `cache` + `results` and goes straight to
training the runs that are still missing. `dataset` only matters for the optional
ResNet table, which builds a second cache at 224.

One caveat: checkpoints are per-run, not per-epoch. A run interrupted midway is
redone from scratch, because resuming mid-run would mean restoring optimizer,
scheduler, and RNG state -- and getting that subtly wrong would silently break the
reproducibility guarantee this project rests on. Losing at most one run is the
cheaper trade.


In [ ]:
import os
import sys

# Re-running this cell in a session where Drive is already mounted makes
# drive.mount() raise "Mountpoint must not already contain files", so check first.
if os.path.isdir("/content/drive/MyDrive"):
    print("Drive already mounted.")
else:
    from google.colab import drive

    drive.mount("/content/drive")

# Two import paths, for two different consumers. `pip install -e .` above makes
# fedswarm importable to the *child* interpreters every stage below shells out to.
# It does nothing for *this* kernel, which started before the install ran and so
# never scanned the new .pth file -- hence the src entry for in-kernel imports.
sys.path.insert(0, "src")

from fedswarm.utils.checkpoint import Checkpoint

ckpt = Checkpoint("/content/drive/MyDrive/fedswarm_backup")
print("restored:", ckpt.restore() or "nothing (first run)")
print()
print(ckpt.report())

## Upload the dataset

Run this cell either way -- it no-ops when the `dataset` stage was restored, and
otherwise opens a file picker. Select the Brain Tumor MRI zip (~164MB), the same
one used locally to build `manifest.csv`. It is saved as `data/raw/dataset.zip`
and checkpointed, so this upload is a one-time cost across all future sessions.


In [ ]:
from pathlib import Path

DATASET_ZIP = Path("data/raw/dataset.zip")

if DATASET_ZIP.exists():
    print(f"{DATASET_ZIP} already present -- skipping the upload.")
else:
    from google.colab import files

    uploaded = files.upload()
    # Colab keys the dict by the name it saved the file under, which collides into
    # things like `archive (1) (2).zip` when a file of that name already exists.
    # Renaming to one predictable path makes the stage restorable by name.
    DATASET_ZIP.parent.mkdir(parents=True, exist_ok=True)
    DATASET_ZIP.write_bytes(next(iter(uploaded.values())))
    print(f"saved {DATASET_ZIP} ({DATASET_ZIP.stat().st_size / 1e6:.1f} MB)")
    print("checkpointed:", ckpt.save("dataset"))

In [ ]:
import os
import subprocess

os.environ["FEDSWARM_DATA_ROOT"] = "/content/brain-tumor-mri"

# Extraction is cheap and derived entirely from the zip, so it is not a checkpointed
# stage of its own -- and it is skipped outright when a restored cache already covers
# it, since training never reads the raw images.
# Gated on the raw dataset root, not the 112 cache -- a resumed session restores
# the 112 cache from Drive but never the extracted raw images (correctly: they're
# derived from the zip, not their own checkpointed stage). Gating on the cache
# instead left FEDSWARM_DATA_ROOT empty on every resumed session, which only broke
# visibly later, in the optional ResNet-18 cell that needs the raw images for a
# second cache at 224.
root = Path(os.environ["FEDSWARM_DATA_ROOT"])
if root.exists() and any(root.iterdir()):
    print(f"{root} already populated -- skipping extraction.")
else:
    # capture_output + print because Jupyter shows nothing a subprocess writes to the
    # kernel's file descriptor; see the training cell for the full explanation.
    proc = subprocess.run(
        ["python", "-m", "fedswarm.data.download",
         "--zip", str(DATASET_ZIP),
         "--root", str(root)],
        capture_output=True, text=True,
    )
    print(proc.stdout, proc.stderr)
    proc.check_returncode()

    # --zip implies --verify, which rewrites the tracked DATASET_CARD.md with this
    # host's scan (different path, fresh timestamp) and never gets committed here --
    # it would just leave every provenance capture for the rest of the session
    # reporting `dirty: true` for a change nobody intends to keep. Revert it.
    subprocess.run(["git", "checkout", "--", "data/raw/DATASET_CARD.md"], check=False)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- check Runtime > Change runtime type before continuing")

## Build the decoded-image cache

Uses the manifest already committed to the repo (`data/processed/manifest.csv`) -- the pseudo-patient-level split from Phase 1.3, with cross-split leakage verified zero. This step is I/O-bound (JPEG decode via PIL), not GPU-bound, and should take well under a minute for 7,200 images.

In [ ]:
import subprocess

# ensure_cache builds only when the array is absent, so this is a no-op on a restored
# cache (the CLI, `python -m fedswarm.data.cache`, always rebuilds by design).
proc = subprocess.run(
    ["python", "-c", "from fedswarm.data.cache import ensure_cache; ensure_cache(112)"],
    capture_output=True, text=True,
)
print(proc.stdout, proc.stderr)
proc.check_returncode()

print("checkpointed:", ckpt.save("cache"))

## Primary ceiling: SimpleCNN @112, 5 seeds x {groupnorm, batchnorm}

GroupNorm is the FL-relevant default and the primary ceiling. The BatchNorm arm run alongside it is a **centralized-only control** -- it shows the accuracy cost of GroupNorm with federation held out. It is *not* the plan's A9 ablation (BatchNorm-vs-GroupNorm aggregation behavior under non-IID federated data), which needs actual federated training and belongs to Phase 4/7.

In [ ]:
import subprocess
import sys
from pathlib import Path


def run_experiment(config_layers, seed, overrides=()):
    """Run one experiment, streaming the child's output into the notebook.

    Jupyter captures writes to Python's sys.stdout, not to the kernel's underlying
    file descriptor -- so a plain subprocess.run() shows nothing in the cell: no
    epoch logs, and no traceback when the child dies. Every failure then surfaces as
    a bare CalledProcessError with the real cause invisible, which is what made four
    unrelated bugs in this notebook so hard to diagnose. Reading the pipe and
    re-printing puts both back where they can be seen.
    """
    cmd = ["python", "scripts/run_experiment.py", "--config", *config_layers, "--seed", str(seed)]
    if overrides:
        cmd += ["--override", *overrides]

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        sys.stdout.flush()
    return proc.wait()


CONFIG = ["configs/base.yaml", "configs/model/simple_cnn.yaml", "configs/experiment/centralized.yaml"]

for norm in ["groupnorm", "batchnorm"]:
    for seed in [0, 1, 2, 3, 4]:
        # A result file is written only once a run completes, so an interrupted run
        # leaves none and is retried, while finished ones are never recomputed.
        if Path(f"results/centralized/simple_cnn_112_{norm}_{seed}.json").exists():
            print(f"skip norm={norm} seed={seed} (already done)")
            continue

        print(f"\n{'='*20} norm={norm} seed={seed} {'='*20}")
        code = run_experiment(CONFIG, seed, [f"model.norm={norm}"])
        if code != 0:
            raise SystemExit(f"run failed (exit {code}) -- see the traceback above")

        # Checkpoint after every run, not at the end of the sweep: the GPU quota can
        # cut the session off at any point, and a runtime recycle wipes /content.
        ckpt.save("results")

print("\n" + ckpt.report())

In [ ]:
!python scripts/summarize_centralized.py

## Optional: secondary ResNet-18 @224 table

"Does the ceiling hold with a larger, pretrained backbone." More expensive than the primary sweep (224px + 11.2M params) -- skip this cell entirely to save time/quota if you just need the primary ceiling.

In [ ]:
# Needs the raw images (a 224 cache cannot be derived from the 112 one), so the
# dataset stage must have been restored or uploaded.
proc = subprocess.run(
    ["python", "-c", "from fedswarm.data.cache import ensure_cache; ensure_cache(224)"],
    capture_output=True, text=True,
)
print(proc.stdout, proc.stderr)
proc.check_returncode()
ckpt.save("cache")

RESNET_CONFIG = ["configs/base.yaml", "configs/model/resnet18.yaml", "configs/experiment/centralized_resnet18.yaml"]

for seed in [0, 1, 2, 3, 4]:
    if Path(f"results/centralized/resnet18_224_groupnorm_{seed}.json").exists():
        print(f"skip resnet18 seed={seed} (already done)")
        continue
    print(f"\n{'='*20} resnet18 seed={seed} {'='*20}")
    code = run_experiment(RESNET_CONFIG, seed)
    if code != 0:
        raise SystemExit(f"run failed (exit {code}) -- see the traceback above")
    ckpt.save("results")

## Getting results back

This zips `results/centralized/*.json` and triggers a browser download directly -- no output-tab hunting like Kaggle. Send the downloaded zip back for analysis.

In [ ]:
from google.colab import files

!cd results && zip -r /content/centralized_results.zip centralized/
files.download("/content/centralized_results.zip")